# 10 — Outcome and score model using historical odds features

This notebook tests whether historical odds features can improve outcome and scoreline models.

Important notes:
- Odds features can improve predictive accuracy.
- They do not, by themselves, prove that a forecasting edge exists.
- Closing or T-1h odds should not be used as features for a T-24h forecast.

Two modes are included:
1. `T24_FEATURES_ONLY`: valid for forecasts made around 24 hours before kickoff.
2. `ALL_MARKET_FEATURES_DIAGNOSTIC`: diagnostic only; may contain leakage for earlier forecasts.

API calls: 0.

Output:
- `data/processed/10_outcome_score_market_features_report_bundle.zip`


In [ ]:
# Cell 1. Imports and helpers.

from pathlib import Path
import json
import re
import zipfile
import warnings

import numpy as np
import pandas as pd

from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression, PoissonRegressor
from sklearn.metrics import accuracy_score, log_loss, mean_absolute_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")

DATA_DIR = Path("data")
PROCESSED_DIR = DATA_DIR / "processed"
JOIN_DIR = PROCESSED_DIR / "join_train_backtest"
OUT_DIR = PROCESSED_DIR / "10_outcome_score_market_features"

OUT_DIR.mkdir(parents=True, exist_ok=True)

RUN_TIMESTAMP_UTC = pd.Timestamp.utcnow().isoformat()
SEED = 42

def save_json(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(payload, f, indent=2, ensure_ascii=False, default=str)
    return path

def multiclass_brier(y_true, proba):
    y_true = np.asarray(y_true).astype(int)
    one_hot = np.zeros_like(proba, dtype=float)
    one_hot[np.arange(len(y_true)), y_true] = 1.0
    return float(np.mean(np.sum((proba - one_hot) ** 2, axis=1)))

def evaluate_probs(name, y_true, proba):
    y_true = np.asarray(y_true).astype(int)
    pred = np.argmax(proba, axis=1)
    return {
        "model": name,
        "n": len(y_true),
        "accuracy": float(accuracy_score(y_true, pred)),
        "log_loss": float(log_loss(y_true, proba, labels=[0, 1, 2])),
        "brier": multiclass_brier(y_true, proba),
    }

def make_logit(C=0.5):
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(
            max_iter=3000,
            C=C,
            random_state=SEED,
        )),
    ])

def make_poisson(alpha=1.0):
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", PoissonRegressor(
            alpha=alpha,
            max_iter=1000,
        )),
    ])

def poisson_score_probs(lambda_home, lambda_away, max_goals=10):
    # Returns 1X2 probabilities and most likely score.
    home = np.arange(max_goals + 1)
    away = np.arange(max_goals + 1)

    ph = np.exp(-lambda_home) * np.power(lambda_home, home) / np.array(
        [np.math.factorial(int(x)) for x in home]
    )

    pa = np.exp(-lambda_away) * np.power(lambda_away, away) / np.array(
        [np.math.factorial(int(x)) for x in away]
    )

    mat = np.outer(ph, pa)
    mat = mat / mat.sum()

    p_home = float(np.tril(mat, -1).sum())
    p_draw = float(np.trace(mat))
    p_away = float(np.triu(mat, 1).sum())

    max_idx = np.unravel_index(np.argmax(mat), mat.shape)
    most_likely_home = int(max_idx[0])
    most_likely_away = int(max_idx[1])

    return {
        "p_away": p_away,
        "p_draw": p_draw,
        "p_home": p_home,
        "most_likely_home_score": most_likely_home,
        "most_likely_away_score": most_likely_away,
    }

print("Setup OK.")


In [ ]:
# Cell 2. Load joined football + market dataset.

joined_path = JOIN_DIR / "training_features_joined_market.csv"

if not joined_path.exists():
    raise FileNotFoundError(
        "Missing data/processed/join_train_backtest/"
        "training_features_joined_market.csv. Run 03 first."
    )

df = pd.read_csv(joined_path)

df["date"] = pd.to_datetime(df["date"], errors="coerce")
df = df.dropna(subset=["date", "outcome"]).copy()
df["outcome"] = df["outcome"].astype(int)

print("df:", df.shape)
display(df.head())


In [ ]:
# Cell 3. Build feature groups.

exclude = {
    "event_id",
    "date",
    "commence_time",
    "home_team",
    "away_team",
    "tournament",
    "outcome",
    "home_score",
    "away_score",
}

numeric_cols = [
    col for col in df.columns
    if col not in exclude
    and pd.api.types.is_numeric_dtype(df[col])
]

market_t24_cols = [
    col for col in numeric_cols
    if col.startswith("T_minus_24h_")
]

market_all_cols = [
    col for col in numeric_cols
    if (
        "market_" in col
        or "_odds" in col
        or "n_bookmakers" in col
        or "move_" in col
    )
]

football_cols = [
    col for col in numeric_cols
    if col not in market_all_cols
]

feature_sets = {
    "football_only": football_cols,
    "market_T24_only": market_t24_cols,
    "football_plus_market_T24": football_cols + market_t24_cols,
    "football_plus_all_market_DIAGNOSTIC": football_cols + market_all_cols,
}

feature_summary = pd.DataFrame([
    {
        "feature_set": name,
        "n_features": len(cols),
        "leakage_safe_for_T24_bet": (
            "DIAGNOSTIC" not in name
            and "all_market" not in name
        ),
    }
    for name, cols in feature_sets.items()
])

feature_summary.to_csv(
    OUT_DIR / "feature_set_summary.csv",
    index=False,
)

display(feature_summary)


In [ ]:
# Cell 4. Chronological train / validation / test split.

df = df.sort_values("date").reset_index(drop=True)

q1 = df["date"].quantile(0.60)
q2 = df["date"].quantile(0.80)

df["split"] = np.where(
    df["date"] <= q1,
    "train",
    np.where(df["date"] <= q2, "validation", "test"),
)

split_counts = df.groupby("split").size().reset_index(name="rows")

split_counts.to_csv(
    OUT_DIR / "split_counts.csv",
    index=False,
)

display(split_counts)


In [ ]:
# Cell 5. Train outcome models.

train_df = df[df["split"] == "train"].copy()
val_df = df[df["split"] == "validation"].copy()
test_df = df[df["split"] == "test"].copy()

outcome_rows = []
prediction_parts = []

for name, features in feature_sets.items():
    if len(features) == 0:
        continue

    model = make_logit(C=0.5)

    model.fit(
        train_df[features],
        train_df["outcome"],
    )

    for split_name, part in [
        ("validation", val_df),
        ("test", test_df),
    ]:
        raw = model.predict_proba(part[features])
        proba = np.zeros((len(part), 3))

        for i, cls in enumerate(model.named_steps["model"].classes_):
            proba[:, int(cls)] = raw[:, i]

        row = evaluate_probs(
            f"{name}_{split_name}",
            part["outcome"].to_numpy(),
            proba,
        )

        row["feature_set"] = name
        row["split"] = split_name
        row["n_features"] = len(features)
        row["leakage_safe_for_T24_bet"] = (
            "DIAGNOSTIC" not in name
            and "all_market" not in name
        )

        outcome_rows.append(row)

        pred = part[[
            "date",
            "home_team",
            "away_team",
            "outcome",
            "home_score",
            "away_score",
            "split",
        ]].copy()

        pred["model"] = name
        pred["p_away"] = proba[:, 0]
        pred["p_draw"] = proba[:, 1]
        pred["p_home"] = proba[:, 2]
        pred["pred_class"] = proba.argmax(axis=1)

        prediction_parts.append(pred)

outcome_metrics = pd.DataFrame(outcome_rows)
outcome_predictions = pd.concat(prediction_parts, ignore_index=True)

outcome_metrics.to_csv(
    OUT_DIR / "outcome_model_metrics.csv",
    index=False,
)

outcome_predictions.to_csv(
    OUT_DIR / "outcome_model_predictions.csv",
    index=False,
)

display(outcome_metrics.sort_values(["split", "log_loss"]))


In [ ]:
# Cell 6. Train simple score models.

# Score model is bonus. It predicts home_score and away_score using
# Poisson regressors. We evaluate exact score and goal MAE.

score_feature_sets = {
    "score_football_only": football_cols,
    "score_football_plus_market_T24": football_cols + market_t24_cols,
}

score_rows = []
score_pred_parts = []

for name, features in score_feature_sets.items():
    if len(features) == 0:
        continue

    home_model = make_poisson(alpha=1.0)
    away_model = make_poisson(alpha=1.0)

    home_model.fit(
        train_df[features],
        train_df["home_score"].clip(lower=0),
    )

    away_model.fit(
        train_df[features],
        train_df["away_score"].clip(lower=0),
    )

    for split_name, part in [
        ("validation", val_df),
        ("test", test_df),
    ]:
        lambda_home = home_model.predict(part[features])
        lambda_away = away_model.predict(part[features])

        lambda_home = np.clip(lambda_home, 0.05, 8.0)
        lambda_away = np.clip(lambda_away, 0.05, 8.0)

        pred_home_score = np.round(lambda_home).astype(int)
        pred_away_score = np.round(lambda_away).astype(int)

        exact_score = (
            (pred_home_score == part["home_score"].astype(int).to_numpy())
            & (pred_away_score == part["away_score"].astype(int).to_numpy())
        )

        mae_home = mean_absolute_error(part["home_score"], lambda_home)
        mae_away = mean_absolute_error(part["away_score"], lambda_away)

        # Also convert Poisson goals to outcome probabilities.
        probs = []
        likely_scores = []

        for lh, la in zip(lambda_home, lambda_away):
            out = poisson_score_probs(lh, la, max_goals=10)
            probs.append([out["p_away"], out["p_draw"], out["p_home"]])
            likely_scores.append((
                out["most_likely_home_score"],
                out["most_likely_away_score"],
            ))

        probs = np.asarray(probs)

        row = evaluate_probs(
            f"{name}_as_outcome_{split_name}",
            part["outcome"].to_numpy(),
            probs,
        )

        row.update({
            "feature_set": name,
            "split": split_name,
            "home_goal_mae": float(mae_home),
            "away_goal_mae": float(mae_away),
            "total_goal_mae": float((mae_home + mae_away) / 2),
            "exact_score_accuracy": float(exact_score.mean()),
        })

        score_rows.append(row)

        pred = part[[
            "date",
            "home_team",
            "away_team",
            "home_score",
            "away_score",
            "outcome",
            "split",
        ]].copy()

        pred["model"] = name
        pred["lambda_home"] = lambda_home
        pred["lambda_away"] = lambda_away
        pred["pred_home_score_round"] = pred_home_score
        pred["pred_away_score_round"] = pred_away_score
        pred["score_exact"] = exact_score.astype(int)
        pred["p_away_from_score"] = probs[:, 0]
        pred["p_draw_from_score"] = probs[:, 1]
        pred["p_home_from_score"] = probs[:, 2]

        score_pred_parts.append(pred)

score_metrics = pd.DataFrame(score_rows)
score_predictions = pd.concat(score_pred_parts, ignore_index=True)

score_metrics.to_csv(
    OUT_DIR / "score_model_metrics.csv",
    index=False,
)

score_predictions.to_csv(
    OUT_DIR / "score_model_predictions.csv",
    index=False,
)

display(score_metrics.sort_values(["split", "log_loss"]))


In [ ]:
# Cell 7. Betting relevance explanation report.

explanation = {
    "can_historical_odds_improve_accuracy": True,
    "why": (
        "Historical odds encode the market's aggregate information. "
        "They often improve outcome probability calibration and accuracy."
    ),
    "why_this_does_not_equal_betting_edge": (
        "If the model uses market odds as features, it may simply learn "
        "to copy the market. To bet profitably, we need model probability "
        "to be better than the price available at bet time."
    ),
    "leakage_rule": (
        "For a T-24h betting strategy, only features known at T-24h "
        "can be used. T-1h or closing odds can be used for evaluation "
        "and CLV, not as training features for T-24h betting."
    ),
    "score_model_warning": (
        "Exact score prediction is much harder and noisier than 1X2. "
        "Use score model mostly to derive goal expectations and sanity checks."
    ),
    "real_money_allowed": False,
}

save_json(
    OUT_DIR / "betting_relevance_explanation.json",
    explanation,
)

display(pd.DataFrame([explanation]))


In [ ]:
# Cell 8. Save zip.

summary = {
    "run_timestamp_utc": RUN_TIMESTAMP_UTC,
    "rows_total": int(len(df)),
    "train_rows": int(len(train_df)),
    "validation_rows": int(len(val_df)),
    "test_rows": int(len(test_df)),
    "outcome_metric_rows": int(len(outcome_metrics)),
    "score_metric_rows": int(len(score_metrics)),
    "real_money_allowed": False,
}

save_json(
    OUT_DIR / "summary.json",
    summary,
)

zip_path = PROCESSED_DIR / "10_outcome_score_market_features_report_bundle.zip"

with zipfile.ZipFile(
    zip_path,
    "w",
    compression=zipfile.ZIP_DEFLATED,
) as zf:
    for file in OUT_DIR.rglob("*"):
        if file.is_file():
            zf.write(file, arcname=file.relative_to(OUT_DIR))

display(pd.DataFrame([summary]))

print("Created:", zip_path.resolve())
print("Report bundle created.")
